# NHS England Maternal Care Pathways Master Pipeline
## Stage 9 - Regression analysis on generalised mixed model

### Compiled by Ethan Phillips and Federica Caretta Cortegiani

In [0]:
import sys
import numpy as np
import pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from pyspark.sql import SparkSession 
from pyspark.sql.functions import when, col, lit, count, max, last, first, min, avg, sum, collect_list, collect_set, countDistinct, to_date, datediff, array_contains, size, udf, format_number, regexp_replace, explode, array, array_union, desc_nulls_last, flatten, split, filter
from pyspark.sql.types import IntegerType, StringType, NumericType, DateType, ArrayType
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, Imputer
from pyspark.ml.stat import Correlation, Summarizer
from pyspark.ml.functions import vector_to_array
from pyspark.sql.window import Window
from pyspark.ml.regression import LinearRegression
from pyspark.ml.regression import GeneralizedLinearRegression as GLR
import pyspark.pandas as ps
import statsmodels.api as sm

%run "/Workspace/Shared/Helper_Methods_EP"
%run "/Workspace/Shared/glmm_pql_mixed_effects_working"

In [0]:
spark = SparkSession.builder.getOrCreate()

In [0]:
read_table_name = "msds_diag_busy_filtered_final_imputed_reduced_timed_ind_ranked_stage"

df_master = spark.read.table(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{read_table_name}")

In [0]:
model = 1
mixed_effect_cols = ["birth_year"] 
mixed_effect_cols += ["ld_hosp_org_site_id"]
mode = "a"
if len(mixed_effect_cols) == 2:
    mode = "b"

In [0]:
reg_dict = {
    "Num_obs": "N",
    "N": "N",
    "AIC (corrected)": "AIC",
    "aic": "AIC",
    "Intercept": "Intercept (base odds)",
    'booking_age_u35': "under 35 (reference)", 
    'booking_age_35_40': "35-40", 
    'booking_age_o40': "over 40",
    'mother_bmi_u185': "BMI < 18.5",
    'mother_bmi_185_249': "BMI 18.5 - 24.9 (reference)", 
    'mother_bmi_250_299': "BMI 25.0 - 29.9", 
    'mother_bmi_o300': "BMI > 30.0", 
    "ever_substance_use": "Ever Substance Use",
    "ever_smoker": "Ever Smoker",
    "disability_ind": "Disability",
    "folic_acid_while_pregnant": "Folic acid during pregnancy",
    "ethnic_white_reg": "White (Reference)",
    "ethnic_black_reg": "Black",
    "ethnic_south_asian_reg": "South Asian",
    "ethnic_mixed_reg": "Mixed ethnicity",
    "ethnic_other_reg": "Other ethnicity",
    "mother_unemployed": "Mother unemployed",
    "partner_unemployed": "Partner unemployed",
    "lang_not_english": "Language not English",
    "deprived_reg": "Socioeconomic deprivation (IMD <= 3)",
    "IMD_Rank_scaled": "IMD Rank (0-100, most to least deprived)",
    "baby_male": "Male (Reference)",
    "baby_female": "Female",
    "previous_pregnancy": "Previous pregnancy",
    "prior_live_birth": "Previous live birth",
    "prior_csection": "Previous c-section",
    "prior_24w_loss": "Previous 24w loss",
    "prior_stillbirth": "Previous stillbirth",
    "Prior_Aborted_Pregnancy": "Previous aborted pregnancy",
    "contacts_attended": "Contacts attended (any mode)",
    "contacts_attended_pct": "Contacts attended percentage",
    'birth_2021': "Birth 2021 (Reference)",
    'birth_2022': "Birth 2022",
    'birth_2023': "Birth 2023",
    'birth_2024': "Birth 2024",
    'birth_2025': "Birth 2025",
    "Prior_Preeclampsia": "Previous preeclampsia",
    "Prior_Gestational_Diabetes": "Previous gestational DM",
    "Prior_Infectious_Disease": "Infectious Diseases",
    "Prior_Neoplasm": "Neoplasms",
    "Prior_Blood_or_Immune_Disease": "Blood or immune diseases",
    "Prior_Endocrine_or_Metabolic_Disease": "Endocrine/Metabolic Diseases",
    "Prior_Mental_Disorder": "Mental Disorders",
    "Prior_Nervous_System_Disease": "Nervous System Diseases",
    "Prior_Circulatory_Disease": "Circulatory Diseases",
    "Prior_Respiratory_Disease": "Respiratory Diseases",
    "Prior_Gastrointestinal_Disease": "Gastrointestinal Diseases",
    "Prior_Musculoskeletal_Disease": "Musculoskeletal Diseases",
    "Prior_Genitourinary_Disease": "Genitourinary Diseases",
    "Prior_Congenital_Abnormality": "Malformations/Abnormalities",
    "delivery_spont_ceph": "Spontaneous cephalic delivery (reference)",
    "delivery_any_csection": "C-Section delivery",
    "delivery_nonstandard_ceph": "Non-spontaneous cephalic delivery",
    "delivery_any_breech": "Breech delivery",
    "birth_weekday": "Birth Weekday (reference)",
    "birth_weekend": "Birth Weekend",
    "birth_winter": "Birth Winter (Jan-Mar) (Reference)",
    "birth_spring": "Birth Spring (Apr-Jun)",
    "birth_summer": "Birth Summer (Jul-Sep)",
    "birth_autumn": "Birth Autumn (Oct-Dec)",
    "birth_am": "Birth AM (reference)",
    "birth_pm": "Birth PM",
    'extremely_preterm_birth': "Extremely Preterm [< 28]", 
    'very_preterm_birth': "Very Preterm [28 - 31]", 
    'moderate_to_late_preterm_birth': "Moderate to Late Preterm [21 - 27]", 
    'not_preterm_birth' : "Not Preterm [>= 37] (reference)",
    'ld_spell_unit_busyness_on_arrival_3mo_p95_quint0': "Quintile 0",
    'ld_spell_unit_busyness_on_arrival_3mo_p95_quint1': "Quintile 1",
    'ld_spell_unit_busyness_on_arrival_3mo_p95_quint2': "Quintile 2 (Reference)",
    'ld_spell_unit_busyness_on_arrival_3mo_p95_quint3': "Quintile 3",
    'ld_spell_unit_busyness_on_arrival_3mo_p95_quint4': "Quintile 4",
    'ld_spell_unit_busyness_before_arrival_3mo_p95_quint0': "Quintile 0 prev",
    'ld_spell_unit_busyness_before_arrival_3mo_p95_quint1': "Quintile 1 prev",
    'ld_spell_unit_busyness_before_arrival_3mo_p95_quint2': "Quintile 2 prev (Reference)",
    'ld_spell_unit_busyness_before_arrival_3mo_p95_quint3': "Quintile 3 prev",
    'ld_spell_unit_busyness_before_arrival_3mo_p95_quint4': "Quintile 4 prev"
}

outcome_dict = {
    "any_preterm": "Any Preterm Birth",
    "major_preterm": "Preterm <32w",
    "birthweight": "Low birthweight",
    "preeclampsia": "Preeclampsia",
    "early_preeclampsia": "Early Preeclampsia (< 34 weeks)",
    "late_preeclampsia": "Late Preeclampsia (>= 34 weeks)",
    "late_preeclampsia_before_LD": "Late Preeclampsia (>= 34 weeks - <= LD+0)",
    "late_preeclampsia_after_LD": "Late Preeclampsia (>= 34 weeks - >= LD+1 & < LD discharge)",
    "gestDM": "Gestational DM",
    "early_gestDM": "Early Gestational DM (< 24 weeks)",
    "late_gestDM": "Late Gestational DM (>= 24 weeks)",
    "ecsection": "Emergency c-section",
    "agpar": "Low APGAR",
    "stillbirth": "Stillbirth",
    "readmit": "Readmission",
    "los": "L&D Log LOS (linear)",
    "neonatal_crit": "Neonatal Critical Incident",
    "nicu": "NICU",
    "neonatal_crit_filled": "Neonatal Critical Incident (filled)",
    "nicu_filled": "NICU (filled)" 
}

In [0]:
reg_m1 = [
    #'deprived_reg'
    'IMD_Rank_scaled'
] #+ ["birth_2022", "birth_2023", "birth_2024", "birth_2025"]

reg_m1_ref = [
    #'deprived_reg'
    'IMD_Rank_scaled'
] #+ ["birth_2021", "birth_2022", "birth_2023", "birth_2024", "birth_2025"]

reg_m2 = reg_m1 + [
    # 'booking_age_u35', (reference)
    'booking_age_35_40', 
    'booking_age_o40',

    #'ethnic_white_reg', (reference)
    'ethnic_black_reg',
    'ethnic_south_asian_reg',
    'ethnic_mixed_reg',
    'ethnic_other_reg',

    'lang_not_english'
]

reg_m2_ref = reg_m1_ref + [
    'booking_age_u35',  # (reference)
    'booking_age_35_40', 
    'booking_age_o40',

    'ethnic_white_reg', #(reference)
    'ethnic_black_reg',
    'ethnic_south_asian_reg',
    'ethnic_mixed_reg',
    'ethnic_other_reg',

    'lang_not_english'
]

reg_m3 = reg_m2 + [
    'mother_bmi_u185', #underweight              
    # 'mother_bmi_185_249', #normal (reference) 
    'mother_bmi_250_299', #overweight 
    'mother_bmi_o300', #obese

    'ever_smoker',
    'folic_acid_while_pregnant',   
    'contacts_attended_pct',

    'Prior_Preeclampsia',
    'Prior_Gestational_Diabetes',
    'Prior_Endocrine_or_Metabolic_Disease',
    'Prior_Mental_Disorder',
    'Prior_Nervous_System_Disease',
    'Prior_Circulatory_Disease',
    'Prior_Respiratory_Disease',
    'Prior_Gastrointestinal_Disease',
    'Prior_Musculoskeletal_Disease',
    'Prior_Congenital_Abnormality'
]

reg_m3_ref = reg_m2_ref + [
    'mother_bmi_u185', #underweight             
    'mother_bmi_185_249', #normal (reference) 
    'mother_bmi_250_299', #overweight 
    'mother_bmi_o300', #obese

    'ever_smoker',
    'folic_acid_while_pregnant', 
    'contacts_attended_pct',

    'Prior_Preeclampsia',
    'Prior_Gestational_Diabetes',
    'Prior_Endocrine_or_Metabolic_Disease',
    'Prior_Mental_Disorder',
    'Prior_Nervous_System_Disease',
    'Prior_Circulatory_Disease',
    'Prior_Respiratory_Disease',
    'Prior_Gastrointestinal_Disease',
    'Prior_Musculoskeletal_Disease',
    'Prior_Congenital_Abnormality'
]

reg_vars = reg_m1
reg_vars_ref = reg_m1_ref
if model == 2:
    reg_vars = reg_m2
    reg_vars_ref = reg_m2_ref
elif model == 3:
    reg_vars = reg_m3
    reg_vars_ref = reg_m3_ref
reg_vars_1 = reg_vars.copy()
reg_vars_2 = reg_vars.copy()
if model == 3:
    reg_vars_1.remove("Prior_Preeclampsia")
    reg_vars_1.remove("Prior_Gestational_Diabetes")

In [0]:
df_master = df_master.filter(col("IMD_Rank").isNotNull())

df_master_group1 = (
    df_master
    .filter(col("group_1") == 1)
    .filter(~((col("Prior_Preeclampsia") == 1) | (col("Prior_Gestational_Diabetes") == 1)))
)

df_master_group2 = (
    df_master
    .filter(col("group_2") == 1)
)

In [0]:
# Define regression datasets for each outcome
df_preeclampsia_1 = df_master_group1.withColumn("label", col("Preeclampsia_during_this_pregnancy"))
df_early_preeclampsia_1 = df_master_group1.withColumn("label", col("Preeclampsia_early_onset"))
df_late_preeclampsia_1 = df_master_group1.withColumn("label", col("Preeclampsia_late_onset"))

df_preeclampsia_2 = df_master_group2.withColumn("label", col("Preeclampsia_during_this_pregnancy"))
df_early_preeclampsia_2 = df_master_group2.withColumn("label", col("Preeclampsia_early_onset"))
df_late_preeclampsia_2 = df_master_group2.withColumn("label", col("Preeclampsia_late_onset"))

# Drop or fill NAs in outcome cols
df_preeclampsia_1 = df_preeclampsia_1.na.drop(subset=['label'])
df_early_preeclampsia_1 = df_early_preeclampsia_1.na.drop(subset=['label'])
df_late_preeclampsia_1 = df_late_preeclampsia_1.na.drop(subset=['label'])

df_preeclampsia_2 = df_preeclampsia_2.na.drop(subset=['label'])
df_early_preeclampsia_2 = df_early_preeclampsia_2.na.drop(subset=['label'])
df_late_preeclampsia_2 = df_late_preeclampsia_2.na.drop(subset=['label'])

In [0]:
print(f"EOPE - Nulliparous: {df_early_preeclampsia_1.filter(col("Preeclampsia_early_onset") == 1).count()} / {df_early_preeclampsia_1.count()}")
print(f"EOPE - Parous: {df_early_preeclampsia_2.filter(col("Preeclampsia_early_onset") == 1).count()} / {df_early_preeclampsia_2.count()}")
print(f"LOPE - Nulliparous: {df_late_preeclampsia_1.filter(col("Preeclampsia_late_onset") == 1).count()} / {df_late_preeclampsia_1.count()}")
print(f"LOPE - Parous: {df_late_preeclampsia_2.filter(col("Preeclampsia_late_onset") == 1).count()} / {df_late_preeclampsia_2.count()}")

In [0]:
def _prepare(df, var_list, group_cols):
    # select feature columns plus label
    df_sel = df.select(*(var_list + ["label"] + group_cols))

    # assemble feature vector
    assembler = VectorAssembler(inputCols=var_list, outputCol="features")
    return assembler.transform(df_sel)#.select(["feature", "outcome"] + group_cols)

df_preeclampsia_1_reg = _prepare(df_preeclampsia_1, reg_vars_1, mixed_effect_cols)
df_early_preeclampsia_1_reg = _prepare(df_early_preeclampsia_1, reg_vars_1, mixed_effect_cols)
df_late_preeclampsia_1_reg = _prepare(df_late_preeclampsia_1, reg_vars_1, mixed_effect_cols)

df_preeclampsia_2_reg = _prepare(df_preeclampsia_2, reg_vars_2, mixed_effect_cols)
df_early_preeclampsia_2_reg = _prepare(df_early_preeclampsia_2, reg_vars_2, mixed_effect_cols)
df_late_preeclampsia_2_reg = _prepare(df_late_preeclampsia_2, reg_vars_2, mixed_effect_cols)

In [0]:
res = fit_logistic_mixed_pql_df_only(
    df_preeclampsia_1_reg,
    group_cols=mixed_effect_cols,
    features_col='features',
    label_col='label',
    include_intercept=True,
    warm_start=True,
    lr_regparam=0.0,
    verbose=True
)

group1_preeclampsia_summary = print_model_summary(
    res,
    feature_names= reg_vars_1 + ["Intercept"],
    include_intercept=True
)

In [0]:
res = fit_logistic_mixed_pql_df_only(
    df_early_preeclampsia_1_reg,
    group_cols=mixed_effect_cols,
    features_col='features',
    label_col='label',
    include_intercept=True,
    warm_start=True,
    lr_regparam=0.0,
    verbose=True
)

group1_early_preeclampsia_summary = print_model_summary(
    res,
    feature_names= reg_vars_1 + ["Intercept"],
    include_intercept=True
)

In [0]:
res = fit_logistic_mixed_pql_df_only(
    df_late_preeclampsia_1_reg,
    group_cols=mixed_effect_cols,
    features_col='features',
    label_col='label',
    include_intercept=True,
    warm_start=True,
    lr_regparam=0.0,
    verbose=True
)

group1_late_preeclampsia_summary = print_model_summary(
    res,
    feature_names= reg_vars_1 + ["Intercept"],
    include_intercept=True
)

In [0]:
res = fit_logistic_mixed_pql_df_only(
    df_preeclampsia_2_reg,
    group_cols=mixed_effect_cols,
    features_col='features',
    label_col='label',
    include_intercept=True,
    warm_start=True,
    lr_regparam=0.0,
    verbose=True
)

group2_preeclampsia_summary = print_model_summary(
    res,
    feature_names= reg_vars_2 + ["Intercept"],
    include_intercept=True
)

In [0]:
res = fit_logistic_mixed_pql_df_only(
    df_early_preeclampsia_2_reg,
    group_cols=mixed_effect_cols,
    features_col='features',
    label_col='label',
    include_intercept=True,
    warm_start=True,
    lr_regparam=0.0,
    verbose=True
)

group2_early_preeclampsia_summary = print_model_summary(
    res,
    feature_names= reg_vars_2 + ["Intercept"],
    include_intercept=True
)

In [0]:
res = fit_logistic_mixed_pql_df_only(
    df_late_preeclampsia_2_reg,
    group_cols=mixed_effect_cols,
    features_col='features',
    label_col='label',
    include_intercept=True,
    warm_start=True,
    lr_regparam=0.0,
    verbose=True
)

group2_late_preeclampsia_summary = print_model_summary(
    res,
    feature_names= reg_vars_2 + ["Intercept"],
    include_intercept=True
)

In [0]:
def compile_summary_alt():
    dfs = {
        ("Group 1", "preeclampsia"): group1_preeclampsia_summary,
        ("Group 1", "early_preeclampsia"): group1_early_preeclampsia_summary,
        ("Group 1", "late_preeclampsia"): group1_late_preeclampsia_summary,
        ("Group 2", "preeclampsia"): group2_preeclampsia_summary,
        ("Group 2", "early_preeclampsia"): group2_early_preeclampsia_summary,
        ("Group 2", "late_preeclampsia"): group2_late_preeclampsia_summary
    }

    master = spark.createDataFrame([(v,) for v in (["Intercept", "aic", "N"] + reg_vars_ref)], ["name"])

    stacked = None
    n_rows = []
    aic_rows = []

    for (group, outcome), obj in dfs.items():

        coef_df = spark.createDataFrame(obj["coefficients"])

        tmp = (
            coef_df
            .select("name", "odds", "p_val", "ci_lo", "ci_hi")
            .withColumn("aic", F.lit(None).cast("double"))  # keep schema consistent
            .withColumn("N", F.lit(None).cast("double"))  # keep schema consistent
            .withColumn("group", F.lit(group))
            .withColumn("outcome", F.lit(outcome))
        )

        stacked = tmp if stacked is None else stacked.unionByName(tmp)

        aic_val = float(obj["model_stats"]["aic"])
        n_val = float(obj["model_stats"]["N"])
        col_name = f"{group}_{outcome}"

        aic_rows.append(("aic", col_name, aic_val))
        n_rows.append(("N", col_name, n_val))

    aic_df = spark.createDataFrame(
        aic_rows,
        ["name", "col_name", "value"]
    )

    aic_wide = (
        aic_df
        .groupBy("name")
        .pivot("col_name")
        .agg(F.round(F.first("value"), 2).cast("string"))
    )

    n_df = spark.createDataFrame(
        n_rows,
        ["name", "col_name", "value"]
    )

    n_wide = (
        n_df
        .groupBy("name")
        .pivot("col_name")
        .agg(F.round(F.first("value"), 0).cast("string"))
    )

    stacked = (
        stacked
        .withColumn(
            "col_name",
            F.concat_ws("_", "group", "outcome")
        )
    )

    stacked = stacked.withColumn(
        "odds_star",
        F.when(F.col("name") == "aic", None)
        .when(F.col("name") == "N", None)
        .when(F.col("odds").isNull(),
            F.lit("NA")
        ).otherwise(
            F.concat(
                F.format_number("odds", 4),
                F.when(F.col("p_val") < 0.01, F.lit("**"))
                .when(F.col("p_val") < 0.05, F.lit("*"))
                .otherwise(F.lit(""))
            )
        )
    )

    stacked = stacked.withColumn(
        "odds_interval",
        F.when(F.col("name") == "aic", None)
        .when(F.col("name") == "N", None)
        .when(F.col("odds").isNull(),
            F.lit("NA")
        ).otherwise(
            F.concat(
                F.format_number("odds", 3),
                F.lit(" ["),
                F.format_number("ci_lo", 3),
                F.lit(" - "),
                F.format_number("ci_hi", 3),
                F.lit("]")
            )
        )
    )

    wide = (
        stacked
        .groupBy("name")
        .pivot("col_name")
        .agg(F.first("odds_interval"))
    )
    wide = wide.unionByName(n_wide, allowMissingColumns=True)
    wide = wide.unionByName(aic_wide, allowMissingColumns=True)

    summary = master.join(wide, on="name", how="left")

    mapping_expr = F.create_map(
        *[F.lit(x) for kv in reg_dict.items() for x in kv]
    )

    summary = summary.withColumn(
        "Regressor",
        mapping_expr.getItem(F.col("name"))
    ).drop("name", "odds")

    or_cols = ['Group 1_preeclampsia', 'Group 1_early_preeclampsia', 'Group 1_late_preeclampsia', 'Group 2_preeclampsia', 'Group 2_early_preeclampsia', 'Group 2_late_preeclampsia']
    summary = summary.select("Regressor", *[
        F.when(F.col(c).isNull(), F.lit("NA"))
        .otherwise(F.col(c))
        .alias(c)
        for c in or_cols
    ])

    summary = summary.toPandas()

    # Split columns into two levels
    top_header = []
    bottom_header = []

    for col in summary.columns:
        if col == "Regressor":
            top_header.append("")
            bottom_header.append("Regressor")
        else:
            group, outcome = col.split("_", 1)
            top_header.append(group)
            bottom_header.append(outcome_dict.get(outcome, outcome))

    file_name = f"/Workspace/Shared/excel_table_output/mixed_regression_results_with_CI_m{model}_{mode}.xlsx"

    # Save as CSV with two header rows
    summary.to_csv(
        file_name,
        index=False,
        header=False
    )

    # Now prepend the two header rows manually
    with open(file_name, "r") as f:
        lines = f.readlines()

    with open(file_name, "w") as f:
        f.write(",".join(top_header) + "\n")
        f.write(",".join(bottom_header) + "\n")
        f.writelines(lines)

    return summary

In [0]:
summary = compile_summary_alt()

display(summary)